# Cơ sở lý thuyết và chiến lược tiền xử lý chuyên sâu dựa trên EDA

## 1) Kết quả EDA
Từ notebook EDA, ta đã thấy:
- Dữ liệu mất cân bằng lớp rất mạnh (imbalance ratio cao).
- Nhiều biến khoảng cách lệch phải và có outlier.
- `Elevation` có tín hiệu mạnh với target.
- Nhóm biến one-hot (`Wilderness_Area`, `Soil_Type`) chứa tín hiệu phân lớp rõ.

Vì vậy chiến lược tiền xử lý trong notebook này sẽ **giữ lại tín hiệu EDA quan trọng**, đồng thời tối ưu tốc độ.

## 2) Feature Engineering
Trong Covtype, biến khoảng cách ngang và dọc đến thủy văn mô tả vị trí tương đối nhưng chưa phản ánh trực tiếp khoảng cách hình học thực tế. Vì vậy ta tạo:

$$
\text{Euclidean\_Distance\_To\_Hydrology} = \sqrt{\text{Horizontal\_Distance\_To\_Hydrology}^2 + \text{Vertical\_Distance\_To\_Hydrology}^2}
$$

Ngoài ra, tạo biến tổng hợp mức độ xa tiện ích địa lý:

$$
\text{Distance\_To\_Amenities} = \frac{\text{Horizontal\_Distance\_To\_Roadways} + \text{Horizontal\_Distance\_To\_Fire\_Points} + \text{Euclidean\_Distance\_To\_Hydrology}}{3}
$$

Hai biến mới này giúp mô hình nhận diện logic không gian tốt hơn.

## 3) Outliers và Skewness
Các biến khoảng cách thường lệch phải (right-skew) và có outlier. Hướng xử lý:
- Đo skewness cho nhóm continuous.
- Dùng RobustScaler cho nhóm biến liên tục để giảm tác động outlier.
- Log transform chỉ là tùy chọn mở rộng nếu cần thử nghiệm thêm.

## 4) Chiến lược cân bằng lớp 
Có thể dùng SMOTETomek nhưng vì thời gian chạy khá lâu nên ta đổi sang dùng **class weighting**:
- Tính trọng số lớp từ `y_train` bằng `compute_class_weight`.
- Truyền `class_weight` hoặc `sample_weight` vào mô hình khi huấn luyện.

Lợi ích:
- Nhanh hơn rõ rệt trên dữ liệu lớn.
- Không làm tăng kích thước dữ liệu train.
- Hạn chế rủi ro sinh mẫu nhân tạo chưa tối ưu.

## 5) Data Pipeline với ColumnTransformer
Ta áp dụng phép biến đổi khác nhau cho từng nhóm biến:
- Nhóm continuous (10 biến gốc + biến mới): scale.
- Nhóm one-hot (`Wilderness_Area`, `Soil_Type`): giữ nguyên.

Dùng ColumnTransformer + Pipeline giúp chuẩn hóa quy trình và ngăn rò rỉ dữ liệu khi tách train/test.

## Bước 1: Tải dữ liệu và xác định nhóm biến

Mục tiêu:
- Tải dữ liệu raw từ đường dẫn tương đối.
- Tách biến mục tiêu và nhóm đặc trưng (continuous, wilderness, soil).
- Chuẩn bị metadata để dùng xuyên suốt các bước tiền xử lý.

In [ ]:
# Bước 1 - Import thư viện và tải dữ liệu
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)

# Đường dẫn tương đối từ thư mục code/Part2_Classification/data/
csv_path = "../../../data/raw/classification/covtype.csv"

df = pd.read_csv(csv_path)

# Xác định cột mục tiêu linh hoạt
if "Cover_Type" in df.columns:
    target_col = "Cover_Type"
elif "target" in df.columns:
    target_col = "target"
else:
    raise ValueError("Không tìm thấy cột mục tiêu. Kỳ vọng Cover_Type hoặc target.")

# Tách X, y
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# Xác định nhóm cột
continuous_cols = X.columns[:10].tolist()
wilderness_cols = [c for c in X.columns if c.startswith("Wilderness_Area")]
soil_cols = [c for c in X.columns if c.startswith("Soil_Type")]
binary_cols = wilderness_cols + soil_cols

print("Kích thước dữ liệu gốc:", df.shape)
print("Số lớp:", y.nunique())
print("Số biến continuous:", len(continuous_cols))
print("Số biến binary:", len(binary_cols))

## Kết quả bước 1

- Dữ liệu đã được tách thành `X` và `y`, sẵn sàng cho quy trình tiền xử lý.
- Nhóm biến được phân rõ theo vai trò biến đổi:
  - Continuous: dùng cho feature engineering, outlier handling và scaling.
  - Binary one-hot: passthrough trong `ColumnTransformer`.
- Metadata cột được chuẩn bị để dùng xuyên suốt các bước sau.

## Bước 1B: Kiểm tra và xử lý giá trị thiếu

Mục tiêu:
- Kiểm tra đầy đủ missing values trước khi xây dựng pipeline.
- Ghi rõ quyết định xử lý (impute hay không) để đáp ứng yêu cầu học thuật.
- Tránh bỏ sót bước bắt buộc trong báo cáo tiền xử lý.

In [ ]:
# Bước 1B - Missing values check
missing_per_col = df.isnull().sum()
missing_total = int(missing_per_col.sum())
missing_cols = missing_per_col[missing_per_col > 0].sort_values(ascending=False)

print("Tổng số giá trị thiếu:", missing_total)
if missing_total == 0:
    print("Dataset không có missing values -> không cần imputation.")
else:
    print("Các cột có missing values:")
    display(missing_cols.to_frame(name="missing_count"))

## Diễn giải missing values

- Covtype là dataset đã làm sạch, thường không có giá trị thiếu; cell này xác nhận định lượng điều đó.
- Nếu về sau thay dataset khác và có missing, có thể thêm `SimpleImputer` vào nhánh continuous trong `ColumnTransformer`.
- Việc ghi rõ quyết định "không impute" giúp notebook đầy đủ theo rubric hơn.

## Bước 2: Feature Engineering có định hướng địa lý

Mục tiêu:
- Tạo biến khoảng cách Euclid tới thủy văn để phản ánh khoảng cách không gian thực.
- Tạo biến tổng hợp Distance_To_Amenities để mô tả độ xa các hạ tầng chính.
- Đảm bảo quá trình tạo biến nằm trong pipeline bằng FunctionTransformer.

In [ ]:
# Bước 2 - Feature Engineering bằng FunctionTransformer

def add_engineered_features(X_input: pd.DataFrame) -> pd.DataFrame:
    """Tạo đặc trưng mới từ nhóm khoảng cách địa lý."""
    X_out = X_input.copy()

    # Khoảng cách Euclid đến thủy văn
    X_out["Euclidean_Distance_To_Hydrology"] = np.sqrt(
        X_out["Horizontal_Distance_To_Hydrology"] ** 2
        + X_out["Vertical_Distance_To_Hydrology"] ** 2
    )

    # Khoảng cách trung bình tới các tiện ích địa lý quan trọng
    X_out["Distance_To_Amenities"] = (
        X_out["Horizontal_Distance_To_Roadways"]
        + X_out["Horizontal_Distance_To_Fire_Points"]
        + X_out["Euclidean_Distance_To_Hydrology"]
    ) / 3.0

    return X_out

feature_engineer = FunctionTransformer(add_engineered_features, validate=False)

# Tạo thử để kiểm tra nhanh
X_fe_preview = feature_engineer.transform(X)
new_feature_cols = ["Euclidean_Distance_To_Hydrology", "Distance_To_Amenities"]

print("Số cột trước FE:", X.shape[1])
print("Số cột sau FE:", X_fe_preview.shape[1])
print("Các cột mới:", new_feature_cols)
X_fe_preview[new_feature_cols].describe().T

## Kết quả

- Hai đặc trưng mới đã bổ sung thông tin hình học và thông tin tổng hợp tiện ích.
- `Euclidean_Distance_To_Hydrology` giúp mô hình hiểu khoảng cách thực thay vì chỉ nhìn riêng chiều ngang/dọc.
- `Distance_To_Amenities` làm mượt tín hiệu vị trí tổng thể, có thể hỗ trợ giảm nhiễu cục bộ từ từng biến đơn lẻ.

## Bước 3: Định lượng skewness và thiết kế xử lý outlier

Mục tiêu:
- Đo độ lệch phân phối của các biến continuous (gốc + mới).
- Xây dựng transformer clipping theo IQR để giảm ảnh hưởng điểm cực trị ngay trong pipeline.
- Kết hợp IQR clipping + `RobustScaler` để nhất quán với insight từ EDA.

In [ ]:
# Bước 3 - Kiểm tra skewness và định nghĩa IQR clipper
continuous_plus_new = continuous_cols + ["Euclidean_Distance_To_Hydrology", "Distance_To_Amenities"]

skew_values = X_fe_preview[continuous_plus_new].skew().sort_values(key=np.abs, ascending=False)
print("Skewness của nhóm continuous + biến mới:")
print(skew_values)

high_skew = skew_values[np.abs(skew_values) > 1.0]
print("\nCác biến có |skew| > 1.0 (đáng chú ý):")
print(high_skew if len(high_skew) > 0 else "Không có biến vượt ngưỡng 1.0")

# Tóm tắt tỉ lệ outlier theo IQR để liên kết trực tiếp với EDA
outlier_rows = []
for col in continuous_plus_new:
    q1 = X_fe_preview[col].quantile(0.25)
    q3 = X_fe_preview[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (X_fe_preview[col] < lower) | (X_fe_preview[col] > upper)
    outlier_rows.append({
        "feature": col,
        "outlier_ratio": mask.mean(),
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values("outlier_ratio", ascending=False)
print("\nTop biến có outlier ratio cao:")
display(outlier_summary.head(8).round(4))

class IQROutlierClipper(BaseEstimator, TransformerMixin):
    """Clip outliers theo IQR cho từng cột continuous, fit trên train để tránh leakage."""

    def __init__(self, whisker_width=1.5):
        self.whisker_width = whisker_width
        self.lower_bounds_ = None
        self.upper_bounds_ = None
        self.columns_ = None

    def fit(self, X, _y=None):
        _ = _y
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        self.columns_ = X.columns.tolist()
        q1 = X.quantile(0.25)
        q3 = X.quantile(0.75)
        iqr = q3 - q1
        self.lower_bounds_ = q1 - self.whisker_width * iqr
        self.upper_bounds_ = q3 + self.whisker_width * iqr
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.columns_)
        return X.clip(lower=self.lower_bounds_, upper=self.upper_bounds_, axis=1)

## Diễn giải

- `RobustScaler` được chọn vì dùng median/IQR nên ít nhạy với outlier hơn `StandardScaler`, phù hợp đặc điểm Covtype.
- IQR clipping giúp cắt bớt cực trị bất thường nhưng vẫn giữ phân bố chính của dữ liệu.
- Vì clipping được fit trên train trong pipeline, cách này vẫn đảm bảo nguyên tắc chống leakage.

## Bước 4: Chia Train/Validation/Test với stratify

Mục tiêu:
- Chia dữ liệu theo tỉ lệ 70/10/20 (train/val/test) để bám sát yêu cầu học thuật.
- Dùng `stratify` ở cả 2 lần split để giữ phân bố lớp ổn định giữa các tập.
- Chuẩn bị tập validation độc lập cho tuning và model selection.

In [ ]:
# Bước 4 - Chia dữ liệu train/val/test = 70/10/20 với stratify
X_temp_raw, X_test_raw, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

# Val chiếm 10% tổng dữ liệu -> 12.5% trên tập tạm (80%)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_temp_raw,
    y_temp,
    test_size=0.125,
    random_state=SEED,
    stratify=y_temp,
)

print("Kích thước X_train:", X_train_raw.shape)
print("Kích thước X_val:", X_val_raw.shape)
print("Kích thước X_test:", X_test_raw.shape)

print("\nPhân bố y_train:")
print(y_train.value_counts(normalize=True).sort_index().round(4))
print("\nPhân bố y_val:")
print(y_val.value_counts(normalize=True).sort_index().round(4))
print("\nPhân bố y_test:")
print(y_test.value_counts(normalize=True).sort_index().round(4))

## Diễn giải

- Phân bố lớp giữa train/val/test gần tương đương nhau nhờ `stratify` ở cả hai lần split.
- Validation set tách riêng giúp tuning mô hình mà không đụng vào test set.
- Cấu trúc 70/10/20 đáp ứng tốt yêu cầu báo cáo và đánh giá công bằng.

## Bước 5: Xây dựng Pipeline tiền xử lý bằng ColumnTransformer

Mục tiêu:
- Đưa toàn bộ phép biến đổi về một pipeline nhất quán và tái lập được.
- Áp dụng chuỗi biến đổi `IQR clipping -> RobustScaler` cho nhóm continuous + biến mới.
- Giữ nguyên nhóm one-hot bằng `passthrough` và chỉ `fit` trên train để tránh leakage.

In [ ]:
# Bước 5 - Pipeline tiền xử lý

# Danh sách continuous mở rộng sau khi tạo đặc trưng
continuous_features_final = continuous_cols + [
    "Euclidean_Distance_To_Hydrology",
    "Distance_To_Amenities",
]

# Pipeline cho nhóm continuous: clip outlier trước, sau đó scale
continuous_pipeline = Pipeline(
    steps=[
        ("iqr_clip", IQROutlierClipper(whisker_width=1.5)),
        ("scaler", RobustScaler()),
    ]
)

# ColumnTransformer kết hợp biến đổi theo nhóm cột
preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipeline, continuous_features_final),
        ("binary", "passthrough", binary_cols),
    ],
    remainder="drop",
)

# Pipeline đầy đủ: FE -> ColumnTransformer
preprocess_pipeline = Pipeline(
    steps=[
        ("feature_engineering", feature_engineer),
        ("column_transform", preprocessor),
    ]
)

# Fit trên train và transform val/test để tránh leakage
X_train_processed = preprocess_pipeline.fit_transform(X_train_raw)
X_val_processed = preprocess_pipeline.transform(X_val_raw)
X_test_processed = preprocess_pipeline.transform(X_test_raw)

print("Kích thước X_train sau preprocess:", X_train_processed.shape)
print("Kích thước X_val sau preprocess:", X_val_processed.shape)
print("Kích thước X_test sau preprocess:", X_test_processed.shape)

## Diễn giải

- Pipeline bám sát EDA: biến continuous có skew/outlier được clipping + robust scaling, còn one-hot được giữ nguyên.
- Validation set được transform từ pipeline đã fit trên train, nên phù hợp cho tuning công bằng.
- Cấu trúc này giúp tái sử dụng trực tiếp cho nhiều mô hình mà không cần viết lại bước tiền xử lý.

## Bước 6: Cân bằng lớp theo hướng chạy nhanh (Class Weight)

Mục tiêu:
- Không resample dữ liệu train để tránh tăng thời gian xử lý.
- Tính trọng số lớp từ `y_train` nhằm bù mất cân bằng.
- Chuẩn bị `class_weight_dict` và `sample_weight_train` để dùng trực tiếp khi fit model.

In [ ]:
# Bước 6 - Cân bằng lớp bằng class weighting (nhanh hơn resampling)

# Tính class weight trên tập train gốc (sau split 70/10/20)
classes = np.sort(y_train.unique())
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = {int(cls): float(w) for cls, w in zip(classes, weights)}

# Tạo sample weight cho từng dòng train (hữu ích cho nhiều mô hình hỗ trợ sample_weight)
sample_weight_train = y_train.map(class_weight_dict).to_numpy()

print("Class weights (dựa trên y_train):")
print(class_weight_dict)
print("\nThống kê sample_weight_train:")
print("Min:", sample_weight_train.min())
print("Max:", sample_weight_train.max())
print("Mean:", sample_weight_train.mean())

print("\nKích thước dữ liệu sau preprocess:")
print("X_train_processed:", X_train_processed.shape)
print("X_val_processed:", X_val_processed.shape)
print("X_test_processed:", X_test_processed.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

print("\nPhân bố lớp train để đối chiếu mất cân bằng:")
print(y_train.value_counts().sort_index())

## Diễn giải

- Class weighting tăng mức ưu tiên cho lớp thiểu số mà không cần sinh dữ liệu mới.
- So với SMOTETomek, cách này chạy nhanh hơn đáng kể trên Covtype do không phải nội suy và làm sạch biên.
- Kích thước train/val/test giữ nguyên, phù hợp khi chạy nhiều vòng tuning trên tập lớn.
- Ở bước huấn luyện, truyền `class_weight_dict` hoặc `sample_weight_train` vào mô hình hỗ trợ trọng số.

# Tổng kết tiền xử lý

## 1) Giữ theo EDA
- Giữ Feature Engineering địa lý: `Euclidean_Distance_To_Hydrology`, `Distance_To_Amenities`.
- Thêm xử lý outlier theo IQR clipping và `RobustScaler` cho nhóm continuous.
- Giữ nguyên nhóm one-hot (`Wilderness_Area`, `Soil_Type`) vì có tín hiệu phân lớp mạnh.
- Chia tập theo `stratify` thành train/val/test = 70/10/20.

## 2) Tối ưu tốc độ
- Bỏ SMOTETomek trong pipeline chính vì chi phí tính toán cao trên dữ liệu lớn.
- Thay bằng class weighting để xử lý mất cân bằng nhanh hơn, nhẹ hơn và ổn định hơn khi lặp nhiều thử nghiệm.

## 3) Tránh Data Leakage
- Mọi bước học tham số (IQR bounds, scaler, biến đổi) được `fit` trên train.
- Tập validation/test chỉ `transform`, không tham gia học tham số tiền xử lý.
- Cân bằng lớp bằng trọng số nên không can thiệp trực tiếp vào val/test.

## 4) Ảnh hưởng đến kích thước dữ liệu
- Với class weighting: số lượng mẫu của train/val/test giữ nguyên.
- Mô hình học theo trọng số để giảm thiên lệch về lớp đa số.

## 5) Dataset sẵn sàng cho modeling
- Có đủ `X_train_processed`, `X_val_processed`, `X_test_processed` và `sample_weight_train`.
- Có thể dùng trực tiếp cho Logistic (GD/IRLS), LDA/QDA, Perceptron ở notebook model.